# Module 14: Reporting, and Work That Outlives You

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

This module has no new method in it. It is about the two things that decide
whether any of the previous thirteen were worth doing: **what you are entitled
to claim**, and **whether anyone can run your analysis again in two years.**

Both are unglamorous. Both are where analyses most often fail, and neither is
recoverable after the fact.

**About 30 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]          # never fit on unfinished months


CALENDAR = pd.period_range("2019-01", "2026-04", freq="M").to_timestamp()


def counts(agency_id):
    """Monthly counts on a complete calendar, so a gap stays visible as missing."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series(d["n_uof"].values, dtype=float,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    s = pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                  index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())
    s = s.reindex(CALENDAR)
    s.index.freq = "MS"
    return s


print(f"{final['agency_id'].nunique()} agencies, {final['year_month'].nunique()} months")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

TREATED = ["A001", "A002", "A004", "A007", "A010"]

f = final.copy()
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]
f["treated"] = f["agency_id"].isin(TREATED).astype(int)
f["lo"] = np.log(f["n_arrests"])
pi = pd.PeriodIndex(f["year_month"], freq="M")
f["t"] = (pi.year.values - 2019) * 12 + pi.month.values - 1
START, FULL = (2023 - 2019) * 12 + 6, (2023 - 2019) * 12 + 10
f["phase"] = ((f["treated"] == 1) & (f["t"] >= START) & (f["t"] < FULL)).astype(float)
f["post"] = ((f["treated"] == 1) & (f["t"] >= FULL)).astype(float)

pct = lambda b: 100 * (np.exp(b) - 1)


def poisson(formula, data):
    return smf.glm(formula, data, family=sm.families.Poisson(),
                   offset=data["lo"]).fit()


clean = f[f["agency_id"] != "A007"]
print(f"{f['agency_id'].nunique()} agencies, {len(f)} agency months")

## 2. Every answer this series gave to one question

The dataset contains one programme with one true effect: a **12 percent
reduction**. Here is every estimate of it produced along the way.

In [ ]:
rows = []

tr = f[f["treated"] == 1]
aft, bef = tr[tr["post"] == 1], tr[(tr["post"] == 0) & (tr["phase"] == 0)]
naive = 100 * ((aft["n_uof"].sum() / aft["n_arrests"].sum())
               / (bef["n_uof"].sum() / bef["n_arrests"].sum()) - 1)
rows.append(("before and after, treated agencies only", f"{naive:+.1f}%", "", "Topic 18"))


def add(label, formula, data, term, where):
    z = poisson(formula, data)
    lo, hi = z.conf_int().loc[term]
    rows.append((label, f"{pct(z.params[term]):+.1f}%",
                 f"[{pct(lo):+.1f}, {pct(hi):+.1f}]", where))


add("agency effects, no month effects",
    "n_uof ~ C(agency_id) + phase + post", clean, "post", "Module 10")
one = f[f["agency_id"] == "A001"].copy()
one["mo"] = pd.PeriodIndex(one["year_month"], freq="M").month
one["sin1"] = np.sin(2 * np.pi * one["mo"] / 12)
one["cos1"] = np.cos(2 * np.pi * one["mo"] / 12)
add("one agency, interrupted series",
    "n_uof ~ I(t/12.0) + sin1 + cos1 + phase + post", one, "post", "Module 11")
add("panel, pre trend violator left in",
    "n_uof ~ C(agency_id) + C(year_month) + phase + post", f, "post", "Module 10")
add("panel, agency and month effects",
    "n_uof ~ C(agency_id) + C(year_month) + phase + post", clean, "post", "Modules 10, 11")

out = pd.DataFrame(rows, columns=["how it was estimated", "estimate",
                                  "95 percent interval", "where"])
print("  the truth is a 12.0 percent reduction\n")
out.set_index("how it was estimated")

**Every one of these was computed correctly.** None involved an error of
arithmetic, a coding bug, or a misapplied formula. They range from a 33 percent
reduction to a 3 percent reduction, and they differ only in which sources of
variation the analyst chose to account for.

That is the argument for the whole series. The difference between the first row
and the last is not skill with software. It is knowing that a secular trend
exists, that one agency was already improving, and that a single agency cannot
resolve an effect this size.

## 3. What you are entitled to claim

| You may say | You may not say |
|---|---|
| the rate fell 12.6 percent relative to comparison agencies | the programme cut use of force by 12.6 percent |
| the interval runs from 17.9 to 6.9 percent | the effect is 12.6 percent |
| pre programme trends were similar after excluding A007 | the parallel trends assumption holds |
| the data cannot distinguish a step from a gradual change | the effect arrived immediately |
| no effect was detected at Orrindale | there was no effect at Orrindale |
| this is consistent with the programme working | this shows the programme works |

The right hand column is not pedantry. Each of those sentences has been written
in a real report, and each one asserts something the analysis did not
establish.

**The single most important sentence in any evaluation report states what the
study could not have detected.** Without it, a null result reads as evidence of
absence, and every reader will make that mistake.

## 4. The report itself

A chief, a council member and a journalist read differently. All three are
served by the same structure, provided the first page is honest.

| Section | Contains | Common failure |
|---|---|---|
| **The headline** | one number, one interval, one window | a point estimate with no interval |
| **The picture** | the series, with the intervention marked | a bar chart of two averages |
| **The comparison** | who the comparison group is and why | "compared to last year" |
| **What was excluded** | every agency, month and record dropped, with reasons | silence |
| **What could not be detected** | the smallest effect the design could have seen | silence |
| **The alternatives** | what else changed at the same time | silence |
| **Reproduction** | where the code and data are | "available on request" |

The four rows marked silence are the ones that get dropped for length. They are
the only rows a careful reader uses to decide whether to believe the first one.

## 5. Numbers in prose

Three rules that survive contact with an audience.

**Give the count next to the rate.** "The rate rose 50 percent" and "it went
from two incidents to three" are the same fact. Only one of them is honest
about the evidence. Beginner Topic 5 is entirely about this.

**Round to the precision you have.** An interval from 17.9 to 6.9 does not
support "12.63 percent". Two significant figures is almost always the limit.

**Name the denominator every time.** "Use of force fell 12 percent" is
ambiguous between counts and rates, and the two are different numbers about
different things. [Module 1](Module_01_Stationarity_Tested.ipynb) opened on
exactly this pair at Ashfell, where the same series tests as not stationary on
the rate and undecided on the count.

In [ ]:
a = final[final["agency_id"] == "A012"]
first, last = a["year_month"].min(), a["year_month"].max()
y0 = a[a["year_month"] < "2020-01"]
y1 = a[a["year_month"] >= "2025-05"]
print(f"  Ashfell, {first} to {last}")
print(f"    counts  {y0['n_uof'].mean():.1f} a month -> {y1['n_uof'].mean():.1f}   "
      f"{100 * (y1['n_uof'].mean() / y0['n_uof'].mean() - 1):+.1f}%")
r0 = 100 * y0["n_uof"].sum() / y0["n_arrests"].sum()
r1 = 100 * y1["n_uof"].sum() / y1["n_arrests"].sum()
print(f"    rate    {r0:.2f} per 100 arrests -> {r1:.2f}   {100 * (r1 / r0 - 1):+.1f}%")

Four percentage points apart, from the same records, over the same months.
The gap is the arrest trend, and nothing more: a rate's trend is the count's
trend minus the denominator's.

Four points is enough to change a sentence. It is also enough that a reader
who is given one number and assumes the other has been misled, and the two
diverge much further at agencies whose arrest volumes moved more.
**Both numbers are correct. A report that prints one without naming which it
is, is not.**

## 6. Work someone else can run

The test is specific: **a colleague with your repository and no access to you
reproduces every number in the report.** Most analyses fail it within a year,
usually because of something small.

| Practice | The failure it prevents |
|---|---|
| Seed every random operation | results that change on rerun |
| Pin library versions | a silent change in a default argument |
| Read data from one canonical file, never from a modified copy | numbers nobody can trace |
| Never edit data by hand | an unrecorded change |
| Put exclusions in code, with a comment giving the reason | "I think we dropped that month" |
| One script that runs end to end | figures that no longer match the text |
| Commit the figures with the code that made them | a chart nobody can rebuild |
| Write the data dictionary before the analysis | a column whose meaning is now a guess |

This repository is arranged that way and it is worth inspecting as an example
rather than a lecture: `Data/generate_synthetic_wadeps.py` is seeded and
produces every CSV, `Data/verify_ground_truth.py` checks that all eleven
planted patterns are still recoverable, and each level's `Figures/make_figures.py`
rebuilds every image from the same source.

In [ ]:
import sys
print("what this notebook actually ran on:")
print(f"  python      {sys.version.split()[0]}")
for mod in ["numpy", "pandas", "statsmodels", "scipy"]:
    try:
        print(f"  {mod:11s} {__import__(mod).__version__}")
    except ImportError:
        print(f"  {mod:11s} not installed")

Print that block in every analysis. It costs four lines and it is the
difference between "the numbers changed and we do not know why" and "the
numbers changed because statsmodels 0.14 altered a default."

## 7. The checklist

Before anything leaves your desk.

**The data**
- [ ] Every excluded record is excluded in code, with a reason in a comment
- [ ] The calendar is complete and missing months are missing, not zero
- [ ] Provisional months are labelled and excluded from fitting
- [ ] Counts and denominators come from the same source and period

**The model**
- [ ] The intervention date was fixed before any estimate was seen
- [ ] Residuals were checked for autocorrelation and for seasonal pattern
- [ ] Dispersion was checked, and any excess was diagnosed rather than absorbed
- [ ] Coefficients you did not care about were read anyway
- [ ] A comparison group exists, and its pre period trend was tested

**The claim**
- [ ] The interval is reported, not just the estimate
- [ ] The window the estimate applies to is stated
- [ ] The smallest detectable effect is stated
- [ ] Counts appear next to rates
- [ ] Nothing causal is claimed that the design cannot support

**The record**
- [ ] One script reproduces every number and figure
- [ ] Versions and seeds are recorded
- [ ] Someone else has run it

## Exercise

Take the last row of the section 2 table and write the paragraph. Then check
it against the checklist.

In [ ]:
# Fill in the blank, then run.
SHOW_A_VERSION = None          # try True

if SHOW_A_VERSION:
    z = poisson("n_uof ~ C(agency_id) + C(year_month) + phase + post", clean)
    lo, hi = z.conf_int().loc["post"]
    n_ag = clean["agency_id"].nunique()
    n_mo = clean["year_month"].nunique()
    print(f"""
    Across {n_ag} agencies and {n_mo} months, use of force at the agencies that
    adopted de escalation training ran {abs(pct(z.params['post'])):.1f} percent below the
    comparison agencies over the {int(clean['post'].sum() / 4)} months after the training was fully
    in place, 95 percent interval from {abs(pct(hi)):.1f} to {abs(pct(lo)):.1f} percent below.
    Rates are incidents per arrest. One agency was excluded because its use of
    force was already falling at 12 percent a year before the programme began,
    against 4 to 5 percent elsewhere, and one month of documented civil unrest
    was excluded. The analysis cannot determine whether the change arrived
    abruptly or built over the first months, and cannot establish that the
    training caused it: the five agencies were selected on their baseline
    rates, and any statewide change affecting them differently from the
    comparison agencies would appear in this estimate.
    """)
else:
    print("Set SHOW_A_VERSION above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

Compare whatever you wrote against the version in the cell, and against the
checklist. The paragraph is 130 words and every clause is doing work.

- the comparison group is named, not implied
- the window is stated, so "12.6 percent" is not floating free
- the denominator is stated
- both exclusions appear, with their reasons and their sizes
- two things the design cannot establish are named explicitly
- the selection mechanism is disclosed, which is the most damaging fact
  available and belongs in the same paragraph as the estimate

What it does not contain is the word "caused", any mention of dynamics, and
any decimal place the interval does not support.

**The hardest sentence to write is the last one**, because it invites the
reader to discount the finding. Write it anyway. A reader who discovers the
selection mechanism on their own discounts the entire report, not one
paragraph of it.

</details>

---

## Where this goes next

Everything in these fourteen modules estimates **association**, described
carefully, with honest uncertainty. The question underneath every one of them
was causal, and none of them answered it.

The [Causal Inference series](../../../Causal_Inference/) takes up what this one
kept deferring: what a counterfactual is, when a comparison group earns the
name, what selection on the outcome does to an estimate, and what can be
claimed when an intervention was not assigned at random.

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*